In [ ]:
from pathlib import Path
import json
import warnings
import sys
from typing import List

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "finetuning_results.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all.columns

In [ ]:
benchmark_order = {
    k: i for i, k in enumerate(BENCHMARK_NAME_MAPPING.keys())
}

In [ ]:
def get_base_model_id(model_id: str) -> str:
    arch_map = {
        'deit_small': 'deit_small_imagenet_full_seed-0',
        'convnext_small': 'convnext_small_imagenet_full_seed-0',
        'resnet50': 'resnet50_imagenet_full',
    }
    return arch_map[model_id.split('-')[0]]

In [ ]:
GROUPBY_COLS = ['model_id', 'benchmark_name', 'seed', 'finetuning_dataset']

def filter_df(df_orig: pd.DataFrame, exp_type:str, groupby_cols: List[str]=GROUPBY_COLS, data_pct:int=None, model_id:str=None, set_base_model_id:bool=True) -> pd.DataFrame:
    df = df_orig.copy()
    df = df[df['exp_type'] == exp_type]
    if data_pct is not None:
        df = df[df['data_pct'] == data_pct]
    if model_id is not None:
        df = df[df['model_id'] == model_id]
        
    df = apply_hiearchical_aggregation(
        df = df,
        groupby_cols = groupby_cols,
        agg_cols = METRICS_COMPACT + ['task_evaluation_acc'],
        agg_func = 'mean'
    )
    
    if set_base_model_id:
        df['base_model_id'] = df['model_id'].apply(get_base_model_id)
    
    return df

In [ ]:
LEFT_ON_COLS = ['base_model_id', 'benchmark_name']
RIGHT_ON_COLS = ['model_id', 'benchmark_name']
SUFFIXES = ('_finetuned', '_baseline')
SORT_BY_COLS = ['finetuning_dataset', 'benchmark_name']

def get_df_diffs(df1: pd.DataFrame, df2: pd.DataFrame, left_on_cols: List[str]=LEFT_ON_COLS, right_on_cols: List[str]=RIGHT_ON_COLS, suffixes: tuple=SUFFIXES, sort_by_cols: List[str]=SORT_BY_COLS) -> pd.DataFrame:
    df_merged = df1.merge(
        right=df2, 
        left_on=left_on_cols, 
        right_on=right_on_cols, 
        suffixes=suffixes
    )
    
    for metric in METRICS_COMPACT + ['task_evaluation_acc']:
        df_merged[f'{metric}_diff'] = df_merged[f'{metric}_finetuned'] - df_merged[f'{metric}_baseline']
    
    sorting_cols = []
    if 'finetuning_dataset' in sort_by_cols:
        sorting_cols.append('finetuning_dataset_order')
        df_merged['finetuning_dataset_order'] = df_merged['finetuning_dataset'].map(benchmark_order)
    if 'benchmark_name' in sort_by_cols:
        sorting_cols.append('benchmark_order')
        df_merged['benchmark_order'] = df_merged['benchmark_name'].map(benchmark_order)

    if sorting_cols:
        df_merged = df_merged.sort_values(
            by=sorting_cols
        )
        
    return df_merged

# Accuracy Drop: ViT-S

In [ ]:
df_finetuned_acc = filter_df(
    df_orig=df_all,
    exp_type='finetuning_data_pct',
    data_pct=100,
    groupby_cols=['model_id', 'finetuning_dataset', 'seed'],
)
df_finetuned_acc.shape

In [ ]:
df_baseline_acc = filter_df(
    df_orig=df_all,
    exp_type='finetuning_baseline',
    model_id='deit_small_imagenet_full_seed-0',
    groupby_cols=['model_id'],
    set_base_model_id=False,
)
df_baseline_acc.shape

In [ ]:
df_diff_ds_acc = get_df_diffs(
    df1=df_finetuned_acc, 
    df2=df_baseline_acc,
    left_on_cols=['base_model_id'],
    right_on_cols=['model_id'],
    sort_by_cols=['finetuning_dataset']
)
df_diff_ds_acc.shape

In [ ]:
zoom = 0.75
figsize = (15,  10)
zoom = 1.0
figsize = (8,  6)
zoom = 0.75
figsize = (8,  6.5)
figsize = (figsize[0] * zoom, figsize[1] * zoom)
fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)


ax = axes

data = df_diff_ds_acc.copy()
data.finetuning_dataset = data.finetuning_dataset.map(BENCHMARK_NAME_MAPPING)

sns.barplot(
    data=data,
    x='finetuning_dataset',
    y='task_evaluation_acc_diff',
    hue='finetuning_dataset',
    palette={v: BENCHMARK_COLORS[k] for k,v in BENCHMARK_NAME_MAPPING.items()},
    ax=ax
)
# ax.axhline(0, color='red', linestyle='--')


# ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
ax.set_title(r'$\Delta$ Task Performance / ViT-S', fontsize=16, fontweight='bold')
ax.set_ylabel(r"$\Delta$ Accuracy", fontsize=12, fontweight='bold')
ax.set_xlabel(f"Finetuning Dataset", fontsize=12, fontweight='bold')


# Rotate x labels for better readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# ax.set_ylim(-0.007, 0.02)
ax.grid(axis='y', linestyle='--', alpha=0.7)

ax.spines[['top', 'right']].set_visible(False)
    
plt.tight_layout()



figures_dir = save_dir
fig_name = 'finetuning-ablation-acc_drop_deitsmall'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

# Architecture Comparison

In [ ]:
df_finetuned_arch = df_all[
    (df_all["exp_type"] == 'finetuning_data_pct') & (df_all["data_pct"] == 100)
    | (df_all["exp_type"] == 'finetuning_architecture')
    ].copy()

df_finetuned_arch = df_finetuned_arch[df_finetuned_arch["data_pct"] == 100]
df_finetuned_arch = apply_hiearchical_aggregation(
    df = df_finetuned_arch,
    groupby_cols = ['model_id', 'seed', 'finetuning_dataset', 'benchmark_name'],
    agg_cols = METRICS_COMPACT + ['task_evaluation_acc'],
    agg_func = 'mean'
)

df_finetuned_arch['base_model_id'] = df_finetuned_arch['model_id'].apply(get_base_model_id)

df_finetuned_arch.shape

In [ ]:
df_baseline_arch = filter_df(
    df_orig=df_all,
    exp_type='finetuning_baseline',
    groupby_cols=['model_id', 'benchmark_name'],
    set_base_model_id=False
)
df_baseline_arch.shape

In [ ]:
df_diff_arch = df_finetuned_arch.merge(
    df_baseline_arch,
    left_on=['base_model_id', 'benchmark_name'],
    right_on=['model_id', 'benchmark_name'],
    suffixes=('_finetuned', '_baseline')
)

for metric in METRICS_COMPACT:
    df_diff_arch[f'{metric}_diff'] = df_diff_arch[f'{metric}_finetuned'] - df_diff_arch[f'{metric}_baseline']

## Add average improvement per finetuning dataset
df_diff_arch_avg = df_diff_arch.groupby(['model_id_baseline', 'base_model_id', 'seed']).agg({
    f'{metric}_diff': 'mean' for metric in METRICS_COMPACT
}).reset_index()
df_diff_arch_avg['finetuning_dataset'] = 'benchmark_average'
df_diff_arch = pd.concat([df_diff_arch, df_diff_arch_avg], ignore_index=True)


df_diff_arch['finetuning_dataset_order'] = df_diff_arch['finetuning_dataset'].map(benchmark_order)
# df_diff_arch['benchmark_order'] = df_diff_arch['benchmark_name'].map(benchmark_order)

df_diff_arch['model_name'] = df_diff_arch.model_id_baseline.map({
    'deit_small_imagenet_full_seed-0': 'ViT-S',
    'convnext_small_imagenet_full_seed-0': 'ConvNeXt-S',
    'resnet50_imagenet_full': 'ResNet-50',
})

df_diff_arch = df_diff_arch.sort_values(
    by=['finetuning_dataset_order', 'model_name']
)

df_diff_arch.finetuning_dataset = df_diff_arch.finetuning_dataset.map(BENCHMARK_NAME_MAPPING)


df_diff_arch.shape

In [ ]:
data = df_diff_arch[df_diff_arch.benchmark_name == 'benchmark_average'].copy()
data

In [ ]:
zoom = 0.75
zoom = 0.6
figsize = (15,  10)
zoom = 1.0
zoom = 0.6
zoom = 2
figsize = (6,  6)
figsize = (10,  9)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

fig, axes = plt.subplots(3, 3, figsize=figsize, dpi=300)

arch_families = df_diff_arch['model_name'].unique()
n_arch = len(arch_families)

palette = {
    'ViT-S': ARCHITECUTURE_FAMILY_COLORS['ViT'],
    'ConvNeXt-S': ARCHITECUTURE_FAMILY_COLORS['ConvNeXt'],
    'ResNet-50': ARCHITECUTURE_FAMILY_COLORS['ResNet'],
}

benchmarks = [
    'bs_fz',
    'bs_mh',
    'tvsd',
    'things_fmri',
    'nsd_func1pt8mm_individualROIs',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
    'benchmark_average',
]

for idx, ds in enumerate(benchmarks):
    ax = axes[idx // 3, idx % 3]
    data = df_diff_arch[df_diff_arch.benchmark_name == ds].copy()
    print(f"Plotting {ds} with {data.shape[0]} data points")
    sns.barplot(
        x='finetuning_dataset',
        y='pearsonr_nc_diff',
        hue='model_name',
        palette=palette,
        # data=df_diff_arch[df_diff_arch.model_id_finetuned.notna()],
        data=data,
        ax=ax
    )

        
    # ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
    ax.set_title(f'{BENCHMARK_NAME_MAPPING[ds]}', fontsize=20, fontweight='bold')
    # ax.set_ylabel(f"Change in Average Alignment", fontsize=16, fontweight='bold')
    ax.set_ylabel(r"$\Delta$ Average Alignment", fontsize=16, fontweight='bold')
    ax.set_xlabel(f"Finetuning Dataset", fontsize=16, fontweight='bold')


    # Rotate x labels for better readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

    # ax.set_ylim(-0.005, 0.015)
    ax.grid(axis='y', linestyle='--', alpha=0.7, zorder=-100)


    # remove legend panel background
    ax.legend(
        # title='Model', 
        title=None, 
        loc='upper center', 
        # frameon=False
    )


    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

# figures_dir = save_dir
# fig_name = 'fig3_finetuning_architecture_pearsonr_nc_diff'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig, figures_dir, fig_name, formats=formats)

# Accuracy Drop Across Architectures

In [ ]:
df_finetuned_arch_acc = df_all[
    (df_all["exp_type"] == 'finetuning_data_pct') & (df_all["data_pct"] == 100)
    | (df_all["exp_type"] == 'finetuning_architecture')
    ].copy()

df_finetuned_arch_acc = df_finetuned_arch_acc[df_finetuned_arch_acc["data_pct"] == 100]
df_finetuned_arch_acc = df_finetuned_arch_acc.groupby([
    'model_id',
    'seed',
    'finetuning_dataset',
]).agg({
    metric: np.nanmean
    for metric in ['task_evaluation_acc']
}).reset_index()
df_finetuned_arch_acc['base_model_id'] = df_finetuned_arch_acc['model_id'].apply(get_base_model_id)

df_finetuned_arch_acc.shape

In [ ]:
df_baseline_arch_acc = filter_df(
    df_orig=df_all,
    exp_type='finetuning_baseline',
    groupby_cols=['model_id'],
    set_base_model_id=False
)
df_baseline_arch_acc.shape

In [ ]:
df_diff_arch_acc = df_finetuned_arch_acc.merge(
    df_baseline_arch_acc,
    left_on=['base_model_id'],
    right_on=['model_id'],
    suffixes=('_finetuned', '_baseline')
)

df_diff_arch_acc['task_evaluation_acc_diff'] = df_diff_arch_acc['task_evaluation_acc_finetuned'] - df_diff_arch_acc['task_evaluation_acc_baseline']

## Add average improvement per finetuning dataset
df_diff_arch_acc_avg = df_diff_arch_acc.groupby(['model_id_baseline', 'base_model_id', 'seed']).agg({
    'task_evaluation_acc_diff': 'mean'
}).reset_index()
df_diff_arch_acc_avg['finetuning_dataset'] = 'average'
df_diff_arch_acc = pd.concat([df_diff_arch_acc, df_diff_arch_acc_avg], ignore_index=True)


df_diff_arch_acc['finetuning_dataset_order'] = df_diff_arch_acc['finetuning_dataset'].map(benchmark_order)
# df_diff_arch_acc['benchmark_order'] = df_diff_arch_acc['benchmark_name'].map(benchmark_order)

df_diff_arch_acc['model_name'] = df_diff_arch_acc.model_id_baseline.map({
    'deit_small_imagenet_full_seed-0': 'ViT-S',
    'convnext_small_imagenet_full_seed-0': 'ConvNeXt-S',
    'resnet50_imagenet_full': 'ResNet-50',
})

df_diff_arch_acc = df_diff_arch_acc.sort_values(
    by=['finetuning_dataset_order', 'model_name']
)


df_diff_arch_acc.finetuning_dataset = df_diff_arch_acc.finetuning_dataset.map(BENCHMARK_NAME_MAPPING)


df_diff_arch_acc.shape

In [ ]:
zoom = 0.75
zoom = 0.6
figsize = (15,  10)
zoom = 0.75
figsize = (8,  6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
ax = axes
sns.barplot(
    x='finetuning_dataset',
    y='task_evaluation_acc_diff',
    hue='model_name',
    data=df_diff_arch_acc,
    palette=palette,
    ax=axes
)

    
# ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
ax.set_title(r'$\Delta$ Task Performance / Models', fontsize=16, fontweight='bold')
# ax.set_ylabel(f"Average Improvement", fontsize=12, fontweight='bold')
ax.set_ylabel(r"$\Delta$ Accuracy", fontsize=16, fontweight='bold')
ax.set_xlabel(f"Finetuning Dataset", fontsize=12, fontweight='bold')


# Rotate x labels for better readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# ax.set_ylim(-0.007, 0.015)
ax.grid(axis='y', linestyle='--', alpha=0.7)

ax.legend(title='Model', loc='lower left')

ax.spines[['top', 'right']].set_visible(False)



figures_dir = save_dir
fig_name = 'finetuning-ablation-acc_drop_archs'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

# Ablation: Random Shuffle

In [ ]:
df_ablation = filter_df(
    df_orig=df_all,
    exp_type='finetuning_ablation',
    groupby_cols=['model_id', 'benchmark_name', 'seed', 'finetuning_dataset', 'random_shuffle'],
)
df_ablation.shape

In [ ]:
df_baseline_ablation = filter_df(
    df_orig=df_all,
    exp_type='finetuning_baseline',
    model_id='deit_small_imagenet_full_seed-0',
    groupby_cols=['model_id', 'benchmark_name'],
    set_base_model_id=False
)
df_baseline_ablation.shape

In [ ]:
df_diff_ablation = df_ablation.merge(
    df_baseline_ablation,
    left_on=['base_model_id', 'benchmark_name'],
    right_on=['model_id', 'benchmark_name'],
    suffixes=('_finetuned', '_baseline')
)

df_diff_ablation['pearsonr_nc_diff'] = df_diff_ablation['pearsonr_nc_finetuned'] - df_diff_ablation['pearsonr_nc_baseline']
df_diff_ablation['rsa_c_test_diff'] = df_diff_ablation['rsa_c_test_finetuned'] - df_diff_ablation['rsa_c_test_baseline']
df_diff_ablation['cka_c_test_diff'] = df_diff_ablation['cka_c_test_finetuned'] - df_diff_ablation['cka_c_test_baseline']

## Add average improvement per finetuning dataset
df_diff_ablation_avg = df_diff_ablation.groupby(['model_id_baseline', 'base_model_id', 'seed', 'finetuning_dataset', 'random_shuffle']).agg({
    'pearsonr_nc_diff': 'mean',
    'rsa_c_test_diff': 'mean',
    'cka_c_test_diff': 'mean'
}).reset_index()
df_diff_ablation_avg['benchmark_name'] = 'average'
df_diff_ablation = pd.concat([df_diff_ablation, df_diff_ablation_avg], ignore_index=True)

df_diff_ablation['finetuning_dataset_order'] = df_diff_ablation['finetuning_dataset'].map(benchmark_order)
df_diff_ablation['benchmark_order'] = df_diff_ablation['benchmark_name'].map(benchmark_order)

df_diff_ablation = df_diff_ablation.sort_values(
    by=['finetuning_dataset_order', 'benchmark_order']
)

df_diff_ablation.shape

### Average

In [ ]:
zoom = 0.75
figsize = (15,  10)
zoom = 1.0
zoom = 0.6
figsize = (8,  7)
figsize = (6,  7)
figsize = (6,  6)

zoom = 0.75
figsize = (8,  6.5)
figsize = (figsize[0] * zoom, figsize[1] * zoom)
fig, axes = plt.subplots(
    1, 1, 
    figsize=figsize,
    # sharex=True,
    sharey=False,
    dpi=300
)


ax = axes

df_subset = df_diff_ablation[df_diff_ablation['benchmark_name'] == 'average'].copy()
df_subset.finetuning_dataset = df_subset.finetuning_dataset.map(BENCHMARK_NAME_MAPPING)
df_subset = df_subset[df_subset['random_shuffle'] == True]

sns.barplot(
    data=df_subset,
    x='finetuning_dataset',
    y='pearsonr_nc_diff',
    hue='finetuning_dataset',
    palette={v: BENCHMARK_COLORS[k] for k,v in BENCHMARK_NAME_MAPPING.items()},
    # hue='random_shuffle',
    ax=ax
)
# ax.axhline(0, color='red', linestyle='--')


# ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
ax.set_title(f'Stimulus-Response Shuffling', fontsize=16, fontweight='bold')
ax.set_ylabel(f"Changing Average Alignment", fontsize=12, fontweight='bold')
ax.set_ylabel(r"$\Delta$ Average Alignment", fontsize=12, fontweight='bold')
ax.set_xlabel(f"Finetuning Dataset", fontsize=12, fontweight='bold')


# Rotate x labels for better readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

ax.set_ylim(-0.1, 0.0)
ax.grid(axis='y', linestyle='--', alpha=0.7)

ax.spines[['top', 'right']].set_visible(False)
    
plt.tight_layout()


# figures_dir = save_dir
# fig_name = 'fig3_finetuning_random_shuffle_ablation'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig, figures_dir, fig_name, formats=formats)


figures_dir = save_dir
fig_name = 'finetuning-ablation-random_shuffle-average'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

### Across Benchmarks

In [ ]:
zoom = 0.75
figsize = (15,  10)
figsize = (figsize[0] * zoom, figsize[1] * zoom)
fig, axes = plt.subplots(
    2, 3, 
    figsize=figsize,
    # sharex=True,
    sharey=True,
    dpi=300
)

datasets = df_diff_ablation['finetuning_dataset']
datasets = datasets[~datasets.isna()].unique()
# remove 'average' from datasets


for i, ds in enumerate(datasets):
    ax = axes.flatten()[i]
    
    df_subset = df_diff_ablation[df_diff_ablation['finetuning_dataset'] == ds].copy()
    df_subset.benchmark_name = df_subset.benchmark_name.map(BENCHMARK_NAME_MAPPING)
    df_subset = df_subset[df_subset['random_shuffle'] == True]
    
    sns.barplot(
        data=df_subset,
        x='benchmark_name',
        y='pearsonr_nc_diff',
        hue='benchmark_name',
        palette={v: BENCHMARK_COLORS[k] for k,v in BENCHMARK_NAME_MAPPING.items()},
        # hue='random_shuffle',
        ax=ax
    )
    # ax.axhline(0, color='red', linestyle='--')
    
    
    # ax.set_title(f'Finetuned on: {benchmark_names[ds]}', fontsize=16, fontweight='bold')
    ax.set_title(f'{BENCHMARK_NAME_MAPPING[ds]}', fontsize=16, fontweight='bold')
    # ax.set_ylabel(f"Changing Average Alignment", fontsize=12, fontweight='bold')
    ax.set_ylabel(r"$\Delta$ Average Alignment", fontsize=12, fontweight='bold')
    ax.set_xlabel(f"Benchmark", fontsize=12, fontweight='bold')
    
    
    # Rotate x labels for better readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    ax.set_ylim(-0.15, 0.03)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    ax.spines[['top', 'right']].set_visible(False)
    
    ax.legend().remove()
    
plt.tight_layout()


# figures_dir = save_dir
# fig_name = 'fig3_finetuning_'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig, figures_dir, fig_name, formats=formats)

figures_dir = save_dir
fig_name = 'finetuning-ablation-random_shuffle-all_benchmarks'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

In [ ]:
# df_diff_ablation[df_diff_ablation['benchmark_name'] == 'average'].copy()

In [ ]:
# df_diff_ablation

# Ablation: # Subjects

In [ ]:
df_subjects = df_all[
    (df_all['exp_type'] == 'finetuning_subject') |
    ((df_all['exp_type'] == 'finetuning_data_pct') & (df_all['data_pct'] == 100) & (df_all['finetuning_dataset'] == 'nsd_func1pt8mm_individualROIs'))
].copy()

subjects_map = {
    f"randsubj_{i}": 
    i for i in range(1, 8)
}
subjects_map['all'] = 8

df_subjects['n_subjects'] = df_subjects.finetuning_subjects.map(subjects_map)
df_subjects.columns
metric = 'pearsonr_nc'

### Average

In [ ]:
zoom = 0.75
figsize = (8,  6)
figsize = (figsize[0] * zoom, figsize[1] * zoom)

fig, axes = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=figsize,
    dpi=300
)


ax = axes
data = df_subjects.copy()
data = apply_hiearchical_aggregation(
    df=data,
    groupby_cols=['n_subjects', 'seed', 'model_id'],
    agg_cols=[metric],
    agg_func='mean'
)
sns.barplot(
    data=data,
    x='n_subjects',
    y=metric,
    color=BENCHMARK_COLORS['nsd_func1pt8mm_individualROIs'],
    # hue='finetuning_dataset',
    # errorbar='sd',
    # errorbar=None,
    # marker='o',
    ax=ax,
    # palette=colors,
)

data_min = data[metric].min()
data_max = data[metric].max()

ax.set_ylim(data_min*0.999, data_max*1.001)


ax.set_title("Effect of Number of Finetuning Subjects", fontsize=16, fontweight='bold')
ax.set_xlabel('#Subjects', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Alignment Score', fontsize=12, fontweight='bold')

ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.spines[['top', 'right']].set_visible(False)


plt.tight_layout()


# figures_dir = save_dir
# fig_name = 'fig3_finetuning_subjects'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig, figures_dir, fig_name, formats=formats)

figures_dir = save_dir
fig_name = 'finetuning-ablation-subjects-average'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)

### Across Benchmarks

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(16, 8),
    dpi=300
)

for i, benchmark_k in enumerate(df_subjects['benchmark_name'].unique()):
    ax = axes[i // 4, i % 4]
    data = df_subjects[df_subjects['benchmark_name'] == benchmark_k].copy()
    data.benchmark_name = data.benchmark_name.map(BENCHMARK_NAME_MAPPING)
    data = apply_hiearchical_aggregation(
        df=data,
        groupby_cols=['n_subjects', 'benchmark_name', 'seed', 'model_id'],
        agg_cols=[metric],
        agg_func='mean'
    )
    # data = data.groupby(
    #     ['n_subjects', 'benchmark_name', 'seed', 'model_id']
    # ).agg(
    #     {metric: 'mean'}
    # ).reset_index()
    sns.barplot(
        data=data,
        x='n_subjects',
        y=metric,
        hue='benchmark_name',
        # errorbar='sd',
        # errorbar=None,
        # marker='o',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k,v in BENCHMARK_NAME_MAPPING.items()}
    )
    
    data_min = data[metric].min()
    data_max = data[metric].max()
    
    ax.set_ylim(data_min*0.999, data_max*1.001)
    
    
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark_k], fontsize=16, fontweight='bold')
    ax.set_xlabel('#Subjects', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score', fontsize=12, fontweight='bold')
    # ax.set_xscale('log')
    
    # if i == 5:
    #     ax.legend(title='Probe Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    # else:
    ax.legend().remove()
    
    ax.spines[['top', 'right']].set_visible(False)
    
plt.tight_layout()

# figures_dir = save_dir
# fig_name = 'fig3_finetuning_subjects-detailed'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig, figures_dir, fig_name, formats=formats)

figures_dir = save_dir
fig_name = 'finetuning-ablation-subjects-all_benchmarks'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)